<a href="https://colab.research.google.com/github/Ashonet/CSCU9M6/blob/main/CSCU9M6_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Environment Setup

In [ ]:
import os, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau, ModelCheckpoint)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, classification_report, ConfusionMatrixDisplay)
from sklearn.preprocessing import LabelEncoder

print(f"TensorFlow  : {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)


TensorFlow  : 2.19.0
GPU available: []


### 1. Dataset Download

In [ ]:
#.csv file paths
TRAIN_CSV = os.path.join("sign_mnist_train.csv")
TEST_CSV  = os.path.join("sign_mnist_test.csv")

# Verify files exist
for path in [TRAIN_CSV, TEST_CSV]:
    if os.path.exists(path):
        print(f"Found: {path}  ({os.path.getsize(path)/1e6:.1f} MB)")
    else:
        print(f"MISSING: {path}")

Found: sign_mnist_train.csv  (83.3 MB)
Found: sign_mnist_test.csv  (21.8 MB)


### Deliverable 1: Data Loading and Preprocessing

### 1.1 Load CSV and Convert to Image Tensors

In [ ]:
# Load raw CSVs
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print(f"Train shape : {train_df.shape}   ({train_df['label'].nunique()} unique labels)")
print(f"Test  shape : {test_df.shape}")
print(f"\nClass distribution (train):")
print(train_df['label'].value_counts().sort_index().to_string())


Train shape : (27455, 785)   (24 unique labels)
Test  shape : (7172, 785)

Class distribution (train):
label
0     1126
1     1010
2     1144
3     1196
4      957
5     1204
6     1090
7     1013
8     1162
10    1114
11    1241
12    1055
13    1151
14    1196
15    1088
16    1279
17    1294
18    1199
19    1186
20    1161
21    1082
22    1225
23    1164
24    1118


In [ ]:
# Fit encoder on all observed labels so mapping is consistent
_le = LabelEncoder()
_le.fit(np.concatenate([train_df['label'].values, test_df['label'].values]))

def parse_df(df):
    labels  = _le.transform(df['label'].values)          # remap to 0-23
    pixels  = df.drop('label', axis=1).values
    images  = pixels.reshape(-1, 28, 28, 1).astype('float32')
    return images, labels

X_train_full, y_train_full = parse_df(train_df)
X_test,       y_test       = parse_df(test_df)

# Update the class-name lookup to use the encoder's ordering
NUM_CLASSES   = len(_le.classes_)                        # 24
CLASS_NAMES   = [chr(ord('A') + raw) for raw in _le.classes_]
LABEL_TO_LETTER = {i: CLASS_NAMES[i] for i in range(NUM_CLASSES)}

print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Pixel range: [{X_train_full.min()}, {X_train_full.max()}]")

### 1.2 Normalise Pixel Values to [0, 1]

In [ ]:
X_train_full = X_train_full / 255.0
X_test       = X_test       / 255.0

print(f"Pixel range after normalisation : [{X_train_full.min():.2f}, {X_train_full.max():.2f}]")